In [1]:
from google.colab import userdata
import os

aws_dir = os.path.expanduser("~/.aws")
os.makedirs(aws_dir, exist_ok=True)

key_id = userdata.get('AWS_ACCESS_KEY_ID')
secret_key = userdata.get('AWS_SECRET_ACCESS_KEY')

with open(f"{aws_dir}/credentials", "w") as f:
    f.write(f"[default]\naws_access_key_id = {key_id}\naws_secret_access_key = {secret_key}\n")

with open(f"{aws_dir}/config", "w") as f:
    f.write("[default]\nregion = us-east-1\n")

print("Successfully configured ~/.aws/credentials and ~/.aws/config")

Successfully configured ~/.aws/credentials and ~/.aws/config


In [2]:
!apt update -q && apt-get install -qy openjdk-21-jdk-headless > /dev/null 2>&1

import subprocess
from IPython.core.magic import register_cell_magic

class PersistentJShell:
    def __init__(self):
        self.proc = subprocess.Popen(
            ["jshell", "-s"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

    def execute(self, code):
        if self.proc.poll() is not None:
            raise RuntimeError("JShell process has terminated.")

        marker = "___JSHELL_CELL_END___"
        payload = code.strip() + f'\nSystem.out.println("{marker}");\n'
        self.proc.stdin.write(payload)
        self.proc.stdin.flush()

        lines = []
        while True:
            line = self.proc.stdout.readline()
            if not line:
                break
            if marker in line:
                break
            lines.append(line)

        return "".join(lines)

_jshell_instance = PersistentJShell()

@register_cell_magic
def jshell(line, cell):
    out = _jshell_instance.execute(cell)
    if out:
        print(out, end="")


Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu noble InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:5 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [73.2 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.3 MB]
Get:11 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,637 kB]
Get:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Packages [52.1 kB]


In [3]:
%%jshell

void runPy(String code) throws Exception {
    String marker = "___END_OF_PYTHON_CELL___";

    // Base64 encode code payload to bypass all quote & newline syntax errors
    String b64 = java.util.Base64.getEncoder().encodeToString(code.getBytes(java.nio.charset.StandardCharsets.UTF_8));

    // Guarantee marker printing using try...finally and display trailing expressions using ast
    String payload = String.format(
        "import base64, ast, traceback\n" +
        "try:\n" +
        "    code_str = base64.b64decode('%s').decode('utf-8')\n" +
        "    tree = ast.parse(code_str)\n" +
        "    if tree.body and isinstance(tree.body[-1], ast.Expr):\n" +
        "        exec(compile(ast.Module(body=tree.body[:-1], type_ignores=[]), '<string>', 'exec'), globals())\n" +
        "        res = eval(compile(ast.Expression(body=tree.body[-1].value), '<string>', 'eval'), globals())\n" +
        "        if res is not None:\n" +
        "            print(res)\n" +
        "    else:\n" +
        "        exec(compile(tree, '<string>', 'exec'), globals())\n" +
        "except Exception:\n" +
        "    traceback.print_exc()\n" +
        "finally:\n" +
        "    print('%s', flush=True)\n",
        b64, marker
    );

    pyIn.println(payload);

    String line;
    while ((line = pyOut.readLine()) != null) {
        if (line.contains(marker)) break;
        System.out.println(line);
    }
}

import java.io.BufferedReader;
import java.io.InputStreamReader;
import java.io.PrintStream;
import java.io.IOException;

Process pythonProcess;
PrintStream pyIn;
BufferedReader pyOut;

try {
    ProcessBuilder pb = new ProcessBuilder("python3", "-i", "-u");
    // Redirect stderr to stdout to prevent pipe deadlock and capture Python errors
    pb.redirectErrorStream(true);
    pythonProcess = pb.start();

    pyIn = new PrintStream(pythonProcess.getOutputStream(), true);
    pyOut = new BufferedReader(new InputStreamReader(pythonProcess.getInputStream()));

    System.out.println("Python interpreter initialized safely.");
} catch (IOException e) {
    e.printStackTrace();
}

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Sep 30, 2026 7:58:11 AM java.util.prefs.FileSystemPreferences$1 run
INFO: Created user preferences directory.
->   ->  ->  ->  ->  ->  ->  ->  ->  ->  ->  ->  Python interpreter initialized safely.


In [4]:
%%jshell

runPy("""
# Google Colab setup: fetch this repository and use this notebook's directory.
from pathlib import Path ; from IPython.display import display, HTML
import os
os.system('pip install boto3');

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():
    os.system('git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming');

NOTEBOOK_DIR = REPO_ROOT / 'assignments/assignment_21'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')
""");

->  Python 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0] on linux
Type "help", "copyright", "credits" or "license" for more information.
>>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Collecting boto3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 5.7 MB/s eta 0:00:00
Cloning into '/content/BITS_programming'...
Updating files:   4% (70/1414)
Updating files:   5% (71/1414)
Updating files:   6% (85/1414)
Updating files:   7% (99/1414)
Updating files:   8% (114/1414)
Updating files:   9% (128/1414)
Updating files:  10% (142/1414)
Updating files:  11% (156/1414)
Updating files:  11% (164/1414)
Updating files:  12% (170/1414)
Updating files:  13% (184/1414)
Updating files:  14% (198/1414)
Updating files:  15% (213/1414)
Updating files:  16% (227/1414)
Updating files:  17% (241/1414)
Updating files:  18%

In [6]:
%%jshell

runPy("""
# Block 2 - Create AWS clients
import boto3

session = boto3.Session()
region  = session.region_name or "us-east-2"

s3           = session.client("s3",         region_name=region)
ecr          = session.client("ecr",        region_name=region)
codebuild    = session.client("codebuild",  region_name=region)
ecs          = session.client("ecs",        region_name=region)
logs_client  = session.client("logs",       region_name=region)
cloudwatch   = session.client("cloudwatch", region_name=region)
sts          = session.client("sts",        region_name=region)
iam          = session.client("iam",        region_name=region)

identity   = sts.get_caller_identity()
account_id = identity["Account"]
caller_arn = identity["Arn"]

# Determine the execution_role_arn
# If it's the root user, directly use caller_arn (not recommended for service roles)
# If it's an assumed role or direct IAM role, use iam.get_role to get the exact ARN.

if caller_arn.endswith(':root'):
    execution_role_arn = caller_arn
    print("Warning: Running as root user. Services requiring a dedicated IAM role (e.g., CodeBuild, ECS) may encounter issues or require specific configurations that are not recommended for root access.")
elif ":assumed-role/" in caller_arn:
    role_name = caller_arn.split(":assumed-role/")[1].split("/")[0]
    # Use iam.get_role() to get the EXACT ARN including /service-role/ path
    execution_role_arn = iam.get_role(RoleName=role_name)["Role"]["Arn"]
elif ":role/" in caller_arn: # Direct IAM role ARN
    role_name = caller_arn.split(":role/")[-1]
    execution_role_arn = iam.get_role(RoleName=role_name)["Role"]["Arn"]
else: # Likely an IAM user
    execution_role_arn = caller_arn
    print("Warning: Running as an IAM user. Services requiring a dedicated IAM role (e.g., CodeBuild, ECS) may encounter issues or require specific configurations. Consider configuring an IAM role.")

print("Region         :", region)
print("Account        :", account_id)
print("Caller ARN     :", caller_arn)
print("Execution role :", execution_role_arn)
""");

->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Warning: Running as root user. Services requiring a dedicated IAM role (e.g., CodeBuild, ECS) may encounter issues or require specific configurations that are not recommended for root access.
Region         : us-east-1
Account        : 455865672536
Caller ARN     : arn:aws:iam::455865672536:root
Execution role : arn:aws:iam::455865672536:root


In [7]:
%%jshell

runPy("""
# Install missing packages
os.system('pip install onnx onnxruntime sagemaker');
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Collecting onnx
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 2.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take 

# Lesson 6 · Model Optimization I — Quantization & Acceleration
### Real AWS Implementation via SageMaker Neo

**Module 05 · AI Platform Engineering · BITS Pilani**

This notebook implements the Lesson 6 concepts against **real AWS services** (no mocks, no local-only pipelines).
The centrepiece is **AWS SageMaker Neo** — the managed model-compilation service that applies INT8 quantization,
graph optimization, and hardware-specific kernel generation in one API call.

## What you'll build

| Step | Action | AWS service used |
|------|--------|------------------|
| 1 | Preflight (imports, region, IAM) | STS, boto3 |
| 2 | Config with **auto-detected PyTorch version** | none |
| 3 | Train a small classifier locally (works around training-quota=0) | none — pure Python |
| 4 | Export to ONNX + dynamic INT8 quantization | onnxruntime (local) |
| 5 | Benchmark FP32 vs ONNX vs ONNX-INT8 (accuracy, latency, size) | none — pure Python |
| 6 | Package + upload PyTorch model to S3 | **S3** |
| 7 | Launch SageMaker Neo compilation job (target = `ml_m5`) | **SageMaker Neo** |
| 8 | Deploy compiled + baseline models to two endpoints | **SageMaker Endpoint** |
| 9 | Smoke test + load test both endpoints | **SageMaker Runtime** |
| 10 | Compare p50/p95 latency + throughput + cost per 1k requests | pandas |
| 11 | Guarded teardown | SageMaker, S3, IAM |

## Cost + time budget

| Resource | Rate | Runtime | Cost |
|---|---|---|---|
| Neo compilation job | included in SageMaker | ~5–8 min | $0.00 |
| 2× ml.m5.large endpoints | $0.14/hr each | ~15 min | ~$0.07 |
| S3 storage | ~$0.023/GB/mo | model artifacts | ~$0.001 |
| **Total** | | **~15 min** | **~$0.08** |

## Prerequisites

- SageMaker Studio JupyterLab in **eu-north-1** (or any region where you have `ml.m5.large` endpoint quota ≥ 2)
- IAM role attached: `AmazonSageMakerAdminIAMExecutionRole` (or equivalent with SageMaker + S3 + IAM read access)
- Kernel: `Python 3` on `SageMaker Distribution 3.x` (has PyTorch 2.x pre-installed)


## 1. Preflight — verify environment before spending AWS budget

Read-only. Confirms package versions, IAM identity, region. Halts with a clear message if anything
is missing so you don't waste time on Section 6+.

In [9]:
%%jshell

runPy("""
# Block 0 - Preflight (read-only, no AWS mutations)
import sys, os, importlib
import boto3
from botocore.exceptions import ClientError

_ok = True

def _check_pkg(name, submodule=None):
    global _ok
    try:
        mod = importlib.import_module(name)
        v = getattr(mod, "__version__", "?")
        if submodule:
            importlib.import_module(f"{name}.{submodule}")
            print(f"  PASS  {name:24s} {v}  (with .{submodule})")
        else:
            print(f"  PASS  {name:24s} {v}")
        return True
    except ImportError as e:
        print(f"  FAIL  {name}{'.' + submodule if submodule else ''}: {e}")
        _ok = False
        return False

print("=== Package versions ===")
_check_pkg("boto3")
_check_pkg("numpy")
_check_pkg("sklearn")
_check_pkg("torch")
_check_pkg("onnx")
_check_pkg("onnxruntime", submodule="quantization")   # need the quantization submodule
_check_pkg("sagemaker")

print("\\n=== AWS identity + region ===")
sts = boto3.client("sts")
try:
    ident = sts.get_caller_identity()
    print(f"  PASS  identity    {ident['Arn']}")
    print(f"  PASS  account_id  {ident['Account']}")
except ClientError as e:
    print(f"  FAIL  cannot get caller identity: {e}")
    _ok = False

_session = boto3.Session()
_region = _session.region_name or os.environ.get("AWS_DEFAULT_REGION")
if _region:
    print(f"  PASS  region      {_region}")
else:
    print(f"  WARN  no default region set. Block 1 will force eu-north-1.")

if not _ok:
    raise RuntimeError("Preflight FAILED - fix rows marked FAIL before proceeding.")
print("\\nPreflight passed. Safe to proceed.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... === Package versions ===
  PASS  boto3                    1.43.105
  PASS  numpy                    2.1.3
  PASS  sklearn                  1.6.1
  PASS  torch                    2.11.0+cpu
  PASS  onnx                     1.23.1
  PASS  onnxruntime              1.30.0  (with .quantization)
  PASS  sagemaker                ?

=== AWS identity + region ===
  PASS  identity    arn:aws:iam::455865672536:root
  PASS  account_id  455865672536
  PASS  region      us-east-1

Preflight passed. Safe to proceed.


## 2. Configuration + PyTorch version auto-detection

Single edit point at the top: `AWS_REGION_NAME` and `ALLOW_AWS_MUTATIONS`.
PyTorch container + Neo framework versions are **derived from your kernel's torch version**
so tracing format matches what Neo expects.

In [12]:
%%jshell

runPy("""
import os, pathlib, configparser

def get_aws_config_cred(profile="default"):
    cred_file = pathlib.Path.home() / ".aws" / "credentials"
    config_file = pathlib.Path.home() / ".aws" / "config"

    parser = configparser.ConfigParser()
    parser.read([cred_file, config_file])

    profile_section = profile if profile in parser else f"profile {profile}"

    if profile_section in parser:
        sec = parser[profile_section]
        key_id = sec.get("aws_access_key_id")
        secret_key = sec.get("aws_secret_access_key")
        session_token = sec.get("aws_session_token")
        if key_id and secret_key:
            return key_id, secret_key, session_token

    return None, None, None

def get_colab_secret(*names, required=True):
    for name in names:
        value = os.environ.get(name)
        if value:
            return value

    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    if userdata is not None:
        for name in names:
            try:
                value = os.environ.get(name) or userdata.get(name)
                if value:
                    return value
            except Exception:
                continue

    if required:
        choices = " or ".join(names)
        raise RuntimeError(
            f"Missing AWS credential. Configure ~/.aws/config or ~/.aws/credentials, "
            f"add a Colab Secret named {choices}, or set environment variables."
        )
    return None

# Block 1 - Configuration + version detection
import json
import torch, datetime as dt
import boto3
from pathlib import Path ; from IPython.display import display, HTML

# --------------------------------------------------------------------
# USER-EDITABLE
# --------------------------------------------------------------------
AWS_REGION_NAME       = "eu-north-1"
ALLOW_AWS_MUTATIONS   = True

PROJECT_NAME          = "lesson6-neo"
ENVIRONMENT           = "demo"
ENDPOINT_INSTANCE     = "ml.m5.large"

# --------------------------------------------------------------------
# AUTO-DETECTED - kernel PyTorch version drives everything downstream
# --------------------------------------------------------------------
_full_torch_ver = torch.__version__.split("+")[0]
_torch_mm       = ".".join(_full_torch_ver.split(".")[:2])

_VERSION_MAP = {
    "2.9": {"pytorch_img": "2.1", "py_img": "py310"},
    "2.8": {"pytorch_img": "2.1", "py_img": "py310"},
    "2.7": {"pytorch_img": "2.1", "py_img": "py310"},
    "2.6": {"pytorch_img": "2.1", "py_img": "py310"},
    "2.5": {"pytorch_img": "2.1", "py_img": "py310"},
    "2.4": {"pytorch_img": "2.1", "py_img": "py310"},
    "2.3": {"pytorch_img": "2.1", "py_img": "py310"},
    "2.2": {"pytorch_img": "2.1", "py_img": "py310"},
    "2.1": {"pytorch_img": "2.1", "py_img": "py310"},
    "2.0": {"pytorch_img": "2.0", "py_img": "py310"},
    "1.13":{"pytorch_img": "1.13","py_img": "py39"},
}
_versions = _VERSION_MAP.get(_torch_mm, {
    "pytorch_img": "2.1", "py_img": "py310"
})
PYTORCH_IMG_VER  = _versions["pytorch_img"]
PY_IMG_VER       = _versions["py_img"]

print(f"Detected kernel PyTorch: {_full_torch_ver}")
print(f"Using PyTorch container: {PYTORCH_IMG_VER} / {PY_IMG_VER}")

# --------------------------------------------------------------------
# Fetch AWS credentials (~/.aws/config -> Colab secrets -> Env)
# --------------------------------------------------------------------
aws_profile = os.environ.get("AWS_PROFILE", "default")
file_key, file_secret, file_token = get_aws_config_cred(aws_profile)

AWS_ACCESS_KEY_ID = file_key or get_colab_secret('AWS_ACCESS_KEY_ID', 'AWS_ACCESS_KEY')
AWS_SECRET_ACCESS_KEY = file_secret or get_colab_secret('AWS_SECRET_ACCESS_KEY', 'AWS_SECRET_KEY')
AWS_SESSION_TOKEN = file_token or get_colab_secret('AWS_SESSION_TOKEN', 'AWS_SECURITY_TOKEN', required=False)

os.environ['AWS_ACCESS_KEY_ID'] = AWS_ACCESS_KEY_ID
os.environ['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET_ACCESS_KEY
if AWS_SESSION_TOKEN:
    os.environ['AWS_SESSION_TOKEN'] = AWS_SESSION_TOKEN

source = f"~/.aws/config or credentials [profile: {aws_profile}]" if file_key else "Colab secrets / Environment variables"
print(f"AWS credentials loaded from {source} and set as environment variables.")

# --------------------------------------------------------------------
# Region + clients
# --------------------------------------------------------------------
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION_NAME
os.environ["AWS_REGION"]         = AWS_REGION_NAME

boto_session  = boto3.Session(region_name=AWS_REGION_NAME)
region        = AWS_REGION_NAME
s3            = boto_session.client("s3")
sm            = boto_session.client("sagemaker")
runtime       = boto_session.client("sagemaker-runtime")
iam           = boto_session.client("iam")
sts           = boto_session.client("sts")

_ident     = sts.get_caller_identity()
account_id = _ident["Account"]
caller_arn = _ident["Arn"]

# Deterministic + unique-per-run resource names
timestamp = dt.datetime.utcnow().strftime("%Y%m%d-%H%M%S")
bucket_name         = f"{PROJECT_NAME}-{account_id}-{region}"
prefix              = f"{PROJECT_NAME}/{ENVIRONMENT}"
model_s3_prefix     = f"{prefix}/models"

model_name_baseline = f"{PROJECT_NAME}-{ENVIRONMENT}-baseline-{timestamp}"
endpoint_baseline   = f"{PROJECT_NAME}-{ENVIRONMENT}-baseline-ep"
config_baseline     = f"{endpoint_baseline}-cfg-{timestamp}"

# Execution role - use the SageMaker default, with fallback
try:
    from sagemaker import get_execution_role
    execution_role = get_execution_role()
except Exception as _e:
    if ":assumed-role/" in caller_arn:
        _role_name = caller_arn.split("/")[1]
        execution_role = f"arn:aws:iam::{account_id}:role/{_role_name}"
    else:
        execution_role = caller_arn
    print(f"Note: sagemaker.get_execution_role() unavailable ({type(_e).__name__}); using {execution_role}")

work_dir = Path("./neo_work")
work_dir.mkdir(exist_ok=True)

print(f"\\nRegion:          {region}")
print(f"Account:         {account_id}")
print(f"Execution role:  {execution_role}")
print(f"S3 bucket:       {bucket_name}")
print(f"Endpoints:       {endpoint_baseline}")
""");

->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Detected kernel PyTorch: 2.11.0
Using PyTorch container: 2.1 / py310
AWS credentials loaded from ~/.aws/config or credentials [profile: default] and set as environment variables.
<string>:131: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
Note: sagemaker.get_execution_role() unavailable (ImportError); using arn:aws:iam::455865672536:root

Region:          eu-north-1
Account:         455865672536
Execution role:  arn:aws:iam::455865672536:root
S3 bucket:       lesson6-neo-455865672536-eu-north-1
Endpoints:       lesson6-neo-demo-baseline-ep


## 3. Ensure S3 bucket exists

Idempotent: creates on first run, no-op on subsequent runs.

In [13]:
%%jshell

runPy("""
# Block 2 - Ensure S3 bucket exists (idempotent)
if ALLOW_AWS_MUTATIONS:
    try:
        if region == "us-east-2":
            s3.create_bucket(Bucket=bucket_name)
        else:
            s3.create_bucket(
                Bucket=bucket_name,
                CreateBucketConfiguration={"LocationConstraint": region},
            )
        print(f"Created bucket: s3://{bucket_name}")
    except ClientError as e:
        code_ = e.response["Error"]["Code"]
        if code_ in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
            print(f"Bucket already exists: s3://{bucket_name}")
        else:
            raise
else:
    print("Skipped (ALLOW_AWS_MUTATIONS=False).")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Created bucket: s3://lesson6-neo-455865672536-eu-north-1


## 4. Train a small classifier locally

**Why local, not SageMaker Training?** Our sandbox has `training job usage = 0 instances` for every ml.* type.
Neo compilation is a *separate* quota bucket that works, so we train in the kernel and hand a ready model
to Neo. In a production account with training quota, you'd use `sagemaker.pytorch.PyTorch` as the estimator
and skip this section.

The model is intentionally tiny — a 2-layer MLP on `make_classification(n_features=20)` — so training
finishes in seconds. The point of the lesson is what happens **after** training.

In [15]:
%%jshell

runPy("""
# Block 3 - Train a small MLP locally (PyTorch)
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# Reproducible synthetic dataset
X, y = make_classification(
    n_samples=5000, n_features=20, n_informative=15,
    n_redundant=3, n_classes=2, random_state=42,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y,
)

# Cast to float32 (Neo cares about explicit dtypes)
X_train = X_train.astype(np.float32)
X_test  = X_test.astype(np.float32)
y_train = y_train.astype(np.int64)
y_test  = y_test.astype(np.int64)

class TabularMLP(nn.Module):
    def __init__(self, n_features=20, n_hidden=64, n_classes=2):
        super().__init__()
        self.fc1 = nn.Linear(n_features, n_hidden)
        self.fc2 = nn.Linear(n_hidden, n_hidden)
        self.fc3 = nn.Linear(n_hidden, n_classes)
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)   # logits (no softmax — endpoint applies softmax later)

torch.manual_seed(42)
model = TabularMLP()
optim_ = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

X_train_t = torch.from_numpy(X_train)
y_train_t = torch.from_numpy(y_train)
X_test_t  = torch.from_numpy(X_test)

model.train()
for epoch in range(30):
    optim_.zero_grad()
    logits = model(X_train_t)
    loss = loss_fn(logits, y_train_t)
    loss.backward()
    optim_.step()
    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            pred = model(X_test_t).argmax(dim=1).numpy()
            acc = accuracy_score(y_test, pred)
            print(f"  epoch {epoch+1:3d}  loss={loss.item():.4f}  test_acc={acc:.4f}")

model.eval()
with torch.no_grad():
    baseline_pred = model(X_test_t).argmax(dim=1).numpy()
baseline_acc = accuracy_score(y_test, baseline_pred)
baseline_f1  = f1_score(y_test, baseline_pred, average="binary")
print(f"\\nBASELINE (FP32 PyTorch)  accuracy={baseline_acc:.4f}  f1={baseline_f1:.4f}")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ...   epoch  10  loss=0.5744  test_acc=0.7090
  epoch  20  loss=0.4654  test_acc=0.7990
  epoch  30  loss=0.3730  test_acc=0.8590

BASELINE (FP32 PyTorch)  accuracy=0.8590  f1=0.8605


## 5. Export to ONNX and apply INT8 quantization

Two things happen here:
1. **Export** the PyTorch model to ONNX — a portable IR that most quantization tools consume.
2. **Dynamic INT8 quantize** with `onnxruntime.quantization` — no calibration data needed, just weights.

This section is **local-only** and serves to teach the concept. The AWS Neo compilation later (Section 7)
consumes the PyTorch traced model directly.

In [16]:
%%jshell

runPy("""
# Block 4 - Export PyTorch -> ONNX (local benchmarking only)
import onnx
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

onnx_fp32_path = work_dir / "model_fp32.onnx"

# Dummy input shape: (batch, features). ONNX needs concrete tensor to trace.
dummy = torch.randn(1, 20, dtype=torch.float32)

torch.onnx.export(
    model,
    dummy,
    onnx_fp32_path.as_posix(),
    input_names  = ["input"],
    output_names = ["logits"],
    dynamic_axes = {"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=13,
    do_constant_folding=True,
    dynamo=False,   # force legacy TorchScript exporter (no onnxscript required)
)

onnx_model = onnx.load(onnx_fp32_path.as_posix())
onnx.checker.check_model(onnx_model)
print(f"ONNX (FP32) exported: {onnx_fp32_path}  size={onnx_fp32_path.stat().st_size:,} bytes")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... ONNX (FP32) exported: neo_work/model_fp32.onnx  size=23,230 bytes


In [17]:
%%jshell

runPy("""
# Block 5 - Apply dynamic INT8 quantization (onnxruntime)
from onnxruntime.quantization import quantize_dynamic, QuantType

onnx_int8_path = work_dir / "model_int8.onnx"

quantize_dynamic(
    model_input  = onnx_fp32_path.as_posix(),
    model_output = onnx_int8_path.as_posix(),
    weight_type  = QuantType.QInt8,
)

fp32_size = onnx_fp32_path.stat().st_size
int8_size = onnx_int8_path.stat().st_size
print(f"ONNX FP32 size: {fp32_size:>8,} bytes")
print(f"ONNX INT8 size: {int8_size:>8,} bytes")
print(f"Compression:    {fp32_size / int8_size:.2f}x smaller  ({(1 - int8_size/fp32_size) * 100:.1f}% reduction)")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... WARNING:root:Please consider to run pre-processing before quantization. Refer to example: https://github.com/microsoft/onnxruntime-inference-examples/blob/main/quantization/image_classification/cpu/ReadMe.md 
ONNX FP32 size:   23,230 bytes
ONNX INT8 size:    9,356 bytes
Compression:    2.48x smaller  (59.7% reduction)


## 6. Local benchmark: FP32 vs ONNX-FP32 vs ONNX-INT8

Measure the three variants **in-kernel** before touching AWS. Latency is per-request p50 over 500 calls,
batch size = 1.

In [19]:
%%jshell

runPy("""
# Block 6 - Local latency + accuracy benchmark
import time
import onnxruntime as ort
import pandas as pd

def bench(session_or_model, X, y, n_runs=500, is_torch=False):
    # Warm up (JIT / kernel selection)
    for i in range(20):
        if is_torch:
            with torch.no_grad():
                _ = session_or_model(torch.from_numpy(X[i:i+1]))
        else:
            _ = session_or_model.run(None, {"input": X[i:i+1]})

    latencies_ms = []
    preds = []
    for i in range(min(n_runs, len(X))):
        t0 = time.perf_counter()
        if is_torch:
            with torch.no_grad():
                logits = session_or_model(torch.from_numpy(X[i:i+1])).numpy()
        else:
            logits = session_or_model.run(None, {"input": X[i:i+1]})[0]
        latencies_ms.append((time.perf_counter() - t0) * 1000)
        preds.append(int(np.argmax(logits, axis=1)[0]))

    return {
        "p50_ms":         np.percentile(latencies_ms, 50),
        "p95_ms":         np.percentile(latencies_ms, 95),
        "p99_ms":         np.percentile(latencies_ms, 99),
        "throughput_rps": 1000.0 / np.mean(latencies_ms),
        "accuracy":       accuracy_score(y[:len(preds)], preds),
    }

# 1. PyTorch FP32 baseline
r1 = bench(model, X_test, y_test, is_torch=True)
r1["size_bytes"] = sum(p.numel() * p.element_size() for p in model.parameters())

# 2. ONNX FP32
sess_fp32 = ort.InferenceSession(onnx_fp32_path.as_posix(), providers=["CPUExecutionProvider"])
r2 = bench(sess_fp32, X_test, y_test)
r2["size_bytes"] = fp32_size

# 3. ONNX INT8 (dynamic quantized)
sess_int8 = ort.InferenceSession(onnx_int8_path.as_posix(), providers=["CPUExecutionProvider"])
r3 = bench(sess_int8, X_test, y_test)
r3["size_bytes"] = int8_size

df_local = pd.DataFrame([r1, r2, r3], index=[
    "PyTorch FP32",
    "ONNX FP32",
    "ONNX INT8 (dynamic)",
])
df_local["size_KB"] = df_local["size_bytes"] / 1024
df_local = df_local[["accuracy", "p50_ms", "p95_ms", "p99_ms", "throughput_rps", "size_KB"]]
df_local = df_local.round({"accuracy": 4, "p50_ms": 3, "p95_ms": 3, "p99_ms": 3,
                            "throughput_rps": 1, "size_KB": 2})
print("=== Local benchmark ===")
print(df_local.to_string())
print()
print(f"INT8 vs FP32 speedup (p50):  {r2['p50_ms'] / r3['p50_ms']:.2f}x")
print(f"INT8 accuracy delta:         {(r3['accuracy'] - r1['accuracy'])*100:+.2f} percentage points")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... === Local benchmark ===
                     accuracy  p50_ms  p95_ms  p99_ms  throughput_rps  size_KB
PyTorch FP32            0.852   0.055   0.071   0.104         17144.2    22.01
ONNX FP32               0.852   0.015   0.020   0.031         64183.6    22.69
ONNX INT8 (dynamic)     0.848   0.020   0.029   0.042         47758.3     9.14

INT8 vs FP32 speedup (p50):  0.74x
INT8 accuracy delta:         -0.40 percentage points


## 7. SageMaker Neo — real AWS compilation

1. Package the **PyTorch traced model** (Neo consumes TorchScript archives)
2. Upload the tarball to S3
3. Launch a Neo compilation job targeting `ml_m5`
4. Wait ~5–8 min for compilation
5. Neo writes the compiled artifact back to S3

Neo internally uses **TVM** + **LLVM** + vendor kernels. From your perspective it's a single API call.

In [20]:
%%jshell

runPy("""
# Block 7 - Package the PyTorch model for Neo
import tarfile

# Trace the model with a fixed input shape
model.eval()
example_input = torch.randn(1, 20, dtype=torch.float32)
traced = torch.jit.trace(model, example_input)
traced_path = work_dir / "model.pth"
traced.save(traced_path.as_posix())

# Package into tar.gz (Neo expects only model.pth at the root)
model_tar_path = work_dir / "model_neo_input.tar.gz"
with tarfile.open(model_tar_path.as_posix(), "w:gz") as tar:
    tar.add(traced_path.as_posix(), arcname="model.pth")

print(f"Packaged: {model_tar_path}  size={model_tar_path.stat().st_size:,} bytes")

s3_input_key = f"{model_s3_prefix}/model_neo_input.tar.gz"
s3.upload_file(model_tar_path.as_posix(), bucket_name, s3_input_key)
model_s3_uri = f"s3://{bucket_name}/{s3_input_key}"
print(f"Uploaded -> {model_s3_uri}")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Packaged: neo_work/model_neo_input.tar.gz  size=25,313 bytes
Uploaded -> s3://lesson6-neo-455865672536-eu-north-1/lesson6-neo/demo/models/model_neo_input.tar.gz


In [22]:
%%jshell

runPy("""
# Create a minimal Neo-only role - clean trust, only the policies Neo needs
import json as _json
import time as _t

neo_role_name = "SageMakerNeoLabRole"
neo_role_arn  = f"arn:aws:iam::{account_id}:role/{neo_role_name}"

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "sagemaker.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

try:
    iam.create_role(
        RoleName=neo_role_name,
        AssumeRolePolicyDocument=_json.dumps(trust_policy),
        Description="Minimal role for SageMaker Neo compilation (lab)",
    )
    print(f"Created role: {neo_role_arn}")
except iam.exceptions.EntityAlreadyExistsException:
    iam.update_assume_role_policy(
        RoleName=neo_role_name, PolicyDocument=_json.dumps(trust_policy),
    )
    print(f"Role exists (trust refreshed): {neo_role_arn}")

# Attach SageMaker + S3 access
for policy_arn in [
    "arn:aws:iam::aws:policy/AmazonSageMakerFullAccess",
    "arn:aws:iam::aws:policy/AmazonS3FullAccess",
]:
    try:
        iam.attach_role_policy(RoleName=neo_role_name, PolicyArn=policy_arn)
        print(f"  Attached: {policy_arn.split('/')[-1]}")
    except iam.exceptions.LimitExceededException:
        print(f"  Skip {policy_arn}: quota (unlikely on new role)")
    except Exception as e:
        print(f"  Skip {policy_arn}: {type(e).__name__}")

print("\\nWaiting 30s for IAM propagation...")
_t.sleep(30)

neo_execution_role = neo_role_arn
print(f"\\nUse this for Neo: {neo_execution_role}")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Role exists (trust refreshed): arn:aws:iam::455865672536:role/SageMakerNeoLabRole
  Attached: AmazonSageMakerFullAccess
  Attached: AmazonS3FullAccess

Waiting 30s for IAM propagation...

Use this for Neo: arn:aws:iam::455865672536:role/SageMakerNeoLabRole


In [23]:
%%jshell

runPy("""
import datetime as dt

# These variables were commented out in an earlier configuration block (0f5fe0c5).
# Re-defining them here to resolve NameError.
neo_output_prefix = f"{prefix}/neo-output"
NEO_TARGET_DEVICE = "ml_m5"
NEO_FRAMEWORK = "PYTORCH"
# Assuming NEO_FW_VERSION should be '2.0' based on PYTORCH_IMG_VER '2.1' fallback from previous cell's commented logic.
NEO_FW_VERSION = "2.0"

# Fresh job name + new role
timestamp = dt.datetime.utcnow().strftime("%Y%m%d-%H%M%S")
compilation_job = f"{PROJECT_NAME}-{ENVIRONMENT}-compile-{timestamp}"

data_input_cfg = json.dumps({"input0": [1, 20]})
neo_output_uri = f"s3://{bucket_name}/{neo_output_prefix}/"

sm.create_compilation_job(
    CompilationJobName=compilation_job,
    RoleArn=neo_execution_role,    # <-- new role, not execution_role
    InputConfig={
        "S3Uri":            model_s3_uri,
        "DataInputConfig":  data_input_cfg,
        "Framework":        NEO_FRAMEWORK,
        "FrameworkVersion": NEO_FW_VERSION,
    },
    OutputConfig={
        "S3OutputLocation": neo_output_uri,
        "TargetDevice":     NEO_TARGET_DEVICE,
    },
    StoppingCondition={"MaxRuntimeInSeconds": 900},
)
print(f"Neo compilation job started: {compilation_job}")
print(f"Using role: {neo_execution_role}")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Neo compilation job started: lesson6-neo-demo-compile-20260930-081634
Using role: arn:aws:iam::455865672536:role/SageMakerNeoLabRole


In [24]:
%%jshell

runPy("""
# Block 7.5 - Handle single concurrent Neo compilation job limit
import time

try:
    # 1. Check status of the target compilation job if it exists
    job_desc = sm.describe_compilation_job(CompilationJobName=compilation_job)
    status = job_desc["CompilationJobStatus"]
    print(f"Compilation job '{compilation_job}' status: {status}")

    # 2. If it is currently running or in progress, wait for it or stop it
    if status in ["INPROGRESS", "STARTING"]:
        print("Existing job is still running. Stopping it to free up concurrency quota...")
        try:
            sm.stop_compilation_job(CompilationJobName=compilation_job)
            time.sleep(10)
        except Exception as e:
            print(f"Could not stop job: {e}")

except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceNotFound":
        print(f"No existing job found with name: {compilation_job}")
    else:
        # Check if another active job is running under the account
        active_jobs = sm.list_compilation_jobs(StatusEquals="INPROGRESS")["CompilationJobSummaries"]
        for j in active_jobs:
            print(f"Stopping active job {j['CompilationJobName']} to release quota...")
            sm.stop_compilation_job(CompilationJobName=j["CompilationJobName"])
        if active_jobs:
            time.sleep(10)
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Compilation job 'lesson6-neo-demo-compile-20260930-081634' status: STARTING
Existing job is still running. Stopping it to free up concurrency quota...


In [25]:
%%jshell

runPy("""
# Block 8 - Launch or retrieve SageMaker Neo compilation job
import time

neo_output_uri = f"s3://{bucket_name}/{neo_output_prefix}/"
data_input_cfg = json.dumps({"input0": [1, 20]})

def launch_compilation_job(job_name):
    print(f"Creating new compilation job: {job_name}")
    sm.create_compilation_job(
        CompilationJobName=job_name,
        RoleArn=neo_execution_role,
        InputConfig={
            "S3Uri": model_s3_uri,
            "DataInputConfig": data_input_cfg,
            "Framework": NEO_FRAMEWORK,
            "FrameworkVersion": NEO_FW_VERSION,
        },
        OutputConfig={
            "S3OutputLocation": neo_output_uri,
            "TargetDevice": NEO_TARGET_DEVICE,
        },
        StoppingCondition={"MaxRuntimeInSeconds": 900},
    )
    print(f"Neo compilation job started: {job_name}")

if ALLOW_AWS_MUTATIONS:
    try:
        # Check existing job status
        job_info = sm.describe_compilation_job(CompilationJobName=compilation_job)
        job_status = job_info["CompilationJobStatus"]
        print(f"Compilation job '{compilation_job}' exists with status: {job_status}")

        if job_status == "COMPLETED":
            compiled_model_s3_uri = job_info["ModelArtifacts"]["S3ModelArtifacts"]
            print(f"Using completed model artifact: {compiled_model_s3_uri}")

        elif job_status in ["INPROGRESS", "STARTING"]:
            print("Job currently running. Waiting for completion...")

        else:
            # For STOPPED or FAILED jobs, generate a new unique job name to bypass quota/conflict
            compilation_job = f"lesson6-neo-demo-compile-{int(time.time())}"
            launch_compilation_job(compilation_job)

    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFound":
            launch_compilation_job(compilation_job)
        else:
            raise e
else:
    print("Skipped.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Compilation job 'lesson6-neo-demo-compile-20260930-081634' exists with status: STOPPED
Creating new compilation job: lesson6-neo-demo-compile-1790756215
Neo compilation job started: lesson6-neo-demo-compile-1790756215


In [28]:
%%jshell

runPy("""
# Block 9 - Poll the compilation job
compiled_s3_uri = None

if ALLOW_AWS_MUTATIONS:
    print(f"Polling {compilation_job} (typical 5-8 min)...")
    while True:
        info = sm.describe_compilation_job(CompilationJobName=compilation_job)
        status = info["CompilationJobStatus"]
        print(f"  {dt.datetime.utcnow().isoformat(timespec='seconds')}  {status}")
        if status in ("COMPLETED", "FAILED", "STOPPED"):
            break
        time.sleep(30)

    if status != "COMPLETED":
        fr = info.get("FailureReason", "(no reason returned)")
        print(f"\\nCompilation ended with {status}")
        print(f"FailureReason: {fr}")
        raise RuntimeError(f"Neo compilation failed - check FailureReason above")

    compiled_s3_uri = info["ModelArtifacts"]["S3ModelArtifacts"]
    print(f"\\nNeo compilation COMPLETED.")
    print(f"Compiled artifact: {compiled_s3_uri}")
else:
    print("Skipped.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Polling lesson6-neo-demo-compile-1790756215 (typical 5-8 min)...
  2026-09-30T08:17:12  STARTING
  2026-09-30T08:17:42  STARTING
  2026-09-30T08:18:12  STARTING
  2026-09-30T08:18:42  INPROGRESS
  2026-09-30T08:19:12  INPROGRESS
  2026-09-30T08:19:43  INPROGRESS
  2026-09-30T08:20:13  COMPLETED

Neo compilation COMPLETED.
Compiled artifact: s3://lesson6-neo-455865672536-eu-north-1/lesson6-neo/demo/neo-output/model_neo_input-ml_m5.tar.gz


## 8. Deploy baseline + Neo-compiled endpoints side by side

Two endpoints so we can benchmark against a fair baseline in the next section.

- **Baseline endpoint**: hosts the *original* traced PyTorch model using SageMaker's stock PyTorch inference container
- **Neo endpoint**: hosts the *Neo-compiled* artifact using SageMaker's Neo runtime container

Both run on `ml.m5.large`. The **only** variable that changes between the two is Neo compilation.

In [29]:
%%jshell

runPy("""
# Block 10 - Look up container image URIs
# Two-path lookup:
#   Path A: SDK-based image_uris.retrieve() - preferred, portable across regions
#   Path B: Hardcoded eu-north-1 URIs - fallback if the SDK path fails
#
# The `from sagemaker import image_uris` form fails on some SDK versions
# (top-level namespace incomplete), so we import the submodule directly.
try:
    import sagemaker.image_uris as _img_uris
    _SDK_LOOKUP = True
except ImportError as _e:
    print(f"SDK image_uris unavailable: {_e}")
    _SDK_LOOKUP = False

def _try_retrieve(framework, version, py_version):
    if not _SDK_LOOKUP:
        return None
    try:
        uri = _img_uris.retrieve(
            framework=framework, region=region,
            version=version, py_version=py_version,
            image_scope="inference", instance_type=ENDPOINT_INSTANCE,
        )
        print(f"  OK   {framework:15s} {version}/{py_version}")
        return uri
    except Exception as e:
        print(f"  FAIL {framework:15s} {version}/{py_version}: {type(e).__name__}: {str(e)[:80]}")
        return None

# --- Baseline (stock PyTorch inference container) ---
print("=== Baseline (stock PyTorch) container lookup ===")
baseline_image = None
for _v, _py in [(PYTORCH_IMG_VER, PY_IMG_VER), ("2.1", "py310"), ("2.0", "py310"),
                ("1.13", "py39"), ("1.12", "py38")]:
    baseline_image = _try_retrieve("pytorch", _v, _py)
    if baseline_image:
        break

if baseline_image is None:
    # Hardcoded eu-north-1 DLC URI (verified working)
    if region == "eu-north-1":
        baseline_image = (
            f"763104351884.dkr.ecr.{region}.amazonaws.com/"
            f"pytorch-inference:2.1.0-cpu-py310-ubuntu20.04-sagemaker"
        )
        print(f"  Using hardcoded eu-north-1 URI: {baseline_image}")
    else:
        raise RuntimeError(
            f"No PyTorch inference container available in {region}. "
            f"Add a hardcoded URI for this region above."
        )

# --- Neo runtime container ---
# print("\\n=== Neo container lookup ===")
# neo_image = None
# for _v, _py in [(NEO_FW_VERSION, PY_IMG_VER), ("2.0", "py310"), ("1.13", "py39"),
#                 ("1.8", "py3"), ("1.7", "py3")]:
#     neo_image = _try_retrieve("neo-pytorch", _v, _py)
#     if neo_image:
#         break

# if neo_image is None:
#     if region == "eu-north-1":
#         neo_image = (
#             f"601324751636.dkr.ecr.{region}.amazonaws.com/"
#             f"sagemaker-inference-pytorch:2.0-cpu-py3"
#         )
#         print(f"  Using hardcoded eu-north-1 URI: {neo_image}")
#     else:
#         raise RuntimeError(
#             f"No Neo-PyTorch runtime container available in {region}. "
#             f"Add a hardcoded URI for this region above."
#         )

print(f"\\nBaseline image: {baseline_image}")
# print(f"Neo image:      {neo_image}")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... SDK image_uris unavailable: No module named 'sagemaker.image_uris'
=== Baseline (stock PyTorch) container lookup ===
  Using hardcoded eu-north-1 URI: 763104351884.dkr.ecr.eu-north-1.amazonaws.com/pytorch-inference:2.1.0-cpu-py310-ubuntu20.04-sagemaker

Baseline image: 763104351884.dkr.ecr.eu-north-1.amazonaws.com/pytorch-inference:2.1.0-cpu-py310-ubuntu20.04-sagemaker


In [30]:
%%jshell

runPy("""
# Block 11 - Write inference.py (supporting Baseline PyTorch & SageMaker Neo DLR) and package model
import tarfile
import json
import os
import numpy as np
import logging

# Configure logging for the script inside the container
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

inference_script = '''import json
import os
import numpy as np
import logging
import tarfile # Added for potential nested tar extraction

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

def model_fn(model_dir):
    logger.info(f"model_fn called with model_dir: {model_dir}")
    logger.info(f"Contents of model_dir: {os.listdir(model_dir)}")

    # Check for nested model.tar.gz, which sometimes occurs with Neo output
    nested_model_tar = os.path.join(model_dir, "model.tar.gz")
    if os.path.exists(nested_model_tar):
        logger.info(f"Found nested model.tar.gz at {nested_model_tar}. Extracting...")
        try:
            with tarfile.open(nested_model_tar, "r:gz") as tar:
                tar.extractall(path=model_dir)
            logger.info(f"Contents of model_dir after extracting nested tar: {os.listdir(model_dir)}")
            os.remove(nested_model_tar) # Clean up the tarball
        except Exception as e:
            logger.error(f"🚨 ERROR: Failed to extract nested model.tar.gz: {e}", exc_info=True)
            raise

    # 1. Check if model was compiled with SageMaker Neo (DLR format)
    compiled_so_path = os.path.join(model_dir, "compiled.so")
    model_json_path = os.path.join(model_dir, "model.json")

    if os.path.exists(compiled_so_path) or os.path.exists(model_json_path):
        logger.info("Detected Neo DLR model artifacts. Attempting to load with DLR.")
        try:
            # Dynamically import dlr to ensure it's available only when needed
            # and to catch import errors if the Neo container is misconfigured
            try:
                import dlr
            except ImportError:
                logger.error("🚨 ERROR: DLR library not found. Ensure it's installed in the Neo container or its path.")
                raise

            model = dlr.Model(model_dir, device="cpu")
            logger.info("DLR model loaded successfully.")
            return model
        except Exception as e:
            logger.error(f"🚨 ERROR: DLR model loading failed: {e}", exc_info=True)
            raise # Re-raise to make sure the worker dies and the error is visible

    # 2. Otherwise load standard PyTorch TorchScript model
    logger.info("Neo DLR model artifacts not found. Attempting to load standard PyTorch model.")
    import torch
    import torch.nn.functional as F

    model_path = os.path.join(model_dir, "model.pth")
    if not os.path.exists(model_path):
        logger.warning(f"model.pth not found in {model_dir}. Searching for other .pt/.pth files.")
        # Fallback check for compiled.pt or any .pt/.pth artifact
        for f in os.listdir(model_dir):
            if f.endswith(".pt") or f.endswith(".pth"):
                model_path = os.path.join(model_dir, f)
                logger.info(f"Found PyTorch model at: {model_path}")
                break
        else:
            logger.error(f"No PyTorch model artifact (.pth or .pt) found in {model_dir}.")
            raise FileNotFoundError(f"No PyTorch model artifact found in {model_dir}")

    model = torch.jit.load(model_path, map_location="cpu")
    model.eval()
    logger.info("PyTorch model loaded successfully.")
    return model

def input_fn(request_body, request_content_type):
    if request_content_type == "application/json":
        payload = json.loads(request_body)
        arr = np.array(payload["instances"], dtype=np.float32)
        return arr
    raise ValueError(f"Unsupported content type: {request_content_type}")

def predict_fn(input_data, model):
    # Handle SageMaker Neo DLR Model
    if hasattr(model, "run"):
        logger.info("Invoking DLR model's run method.")
        logits = model.run(input_data)
        if isinstance(logits, list):
            logits = logits[0]
        exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
        probs = (exp_logits / np.sum(exp_logits, axis=1, keepdims=True)).tolist()
        preds = np.argmax(logits, axis=1).tolist()
        return {"predictions": preds, "probabilities": probs}

    # Handle Standard PyTorch TorchScript Model
    logger.info("Invoking PyTorch model.")
    import torch
    import torch.nn.functional as F

    tensor_input = torch.from_numpy(input_data) if isinstance(input_data, np.ndarray) else input_data
    with torch.no_grad():
        logits = model(tensor_input)
        probs = F.softmax(logits, dim=1).numpy().tolist()
        preds = logits.argmax(dim=1).numpy().tolist()
    return {"predictions": preds, "probabilities": probs}

def output_fn(prediction, response_content_type):
    return json.dumps(prediction), "application/json"'''

script_path = work_dir / "inference.py"
script_path.write_text(inference_script)

# Package model.pth + code/inference.py in one tar
baseline_tar_path = work_dir / "model_baseline.tar.gz"
with tarfile.open(baseline_tar_path.as_posix(), "w:gz") as tar:
    tar.add(traced_path.as_posix(), arcname="model.pth")
    tar.add(script_path.as_posix(), arcname="code/inference.py")

s3_baseline_key = f"{model_s3_prefix}/model_baseline.tar.gz"
s3.upload_file(baseline_tar_path.as_posix(), bucket_name, s3_baseline_key)
baseline_s3_uri = f"s3://{bucket_name}/{s3_baseline_key}"
print(f"Baseline artifact uploaded -> {baseline_s3_uri}")
print(f"Contents: model.pth + code/inference.py (container auto-detects code/)")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Baseline artifact uploaded -> s3://lesson6-neo-455865672536-eu-north-1/lesson6-neo/demo/models/model_baseline.tar.gz
Contents: model.pth + code/inference.py (container auto-detects code/)


In [31]:
%%jshell

runPy("""
# Hardcoded DLC image URIs for eu-north-1 (SageMaker Deep Learning Containers)
# Ref: https://github.com/aws/deep-learning-containers/blob/master/available_images.md
_DLC_ACCOUNT = "763104351884"   # SageMaker DLC account for eu-north-1

baseline_image = (
    f"{_DLC_ACCOUNT}.dkr.ecr.{region}.amazonaws.com/"
    f"pytorch-inference:2.1.0-cpu-py310-ubuntu20.04-sagemaker"
)

# Neo runtime image for eu-north-1
# _NEO_ACCOUNT = "601324751636"   # SageMaker Neo runtime account for eu-north-1
# neo_image = (
#     f"{_NEO_ACCOUNT}.dkr.ecr.{region}.amazonaws.com/"
#     f"sagemaker-inference-pytorch:2.0-cpu-py3"
# )

print(f"Baseline image: {baseline_image}")
# print(f"Neo image:      {neo_image}")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Baseline image: 763104351884.dkr.ecr.eu-north-1.amazonaws.com/pytorch-inference:2.1.0-cpu-py310-ubuntu20.04-sagemaker


In [32]:
%%jshell

runPy("""
# DEFINITIVE FIX: fresh purpose-built role for CreateModel + endpoints
import json as _json, time as _t

fresh_role = "Lesson6CleanExecRole"
fresh_arn  = f"arn:aws:iam::{account_id}:role/{fresh_role}"

# Trust: sagemaker service principal + all 4 STS actions endpoints need, NO conditions
trust = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "sagemaker.amazonaws.com"},
        "Action": ["sts:AssumeRole", "sts:TagSession", "sts:SetContext", "sts:SetSourceIdentity"],
    }],
}

try:
    iam.create_role(RoleName=fresh_role, AssumeRolePolicyDocument=_json.dumps(trust),
                    Description="Clean role for Lesson 6 CreateModel + endpoints")
    print(f"Created {fresh_arn}")
except iam.exceptions.EntityAlreadyExistsException:
    iam.update_assume_role_policy(RoleName=fresh_role, PolicyDocument=_json.dumps(trust))
    print(f"Reset trust on existing {fresh_arn}")

# Permissions: SageMaker + S3 (managed policies — clean role has 0/10, no quota issue)
for arn in ["arn:aws:iam::aws:policy/AmazonSageMakerFullAccess",
            "arn:aws:iam::aws:policy/AmazonS3FullAccess"]:
    try:
        iam.attach_role_policy(RoleName=fresh_role, PolicyArn=arn)
        print(f"  attached {arn.split('/')[-1]}")
    except Exception as e:
        print(f"  {arn.split('/')[-1]}: {type(e).__name__}")

print("Waiting 90s for role + trust propagation...")
_t.sleep(90)

# Point the notebook at the new role for everything downstream
execution_role = fresh_arn
print(f"\\nexecution_role is now: {execution_role}")
print("Re-run Block 12, then Block 13, then Block 14.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Reset trust on existing arn:aws:iam::455865672536:role/Lesson6CleanExecRole
  attached AmazonSageMakerFullAccess
  attached AmazonS3FullAccess
Waiting 90s for role + trust propagation...

execution_role is now: arn:aws:iam::455865672536:role/Lesson6CleanExecRole
Re-run Block 12, then Block 13, then Block 14.


In [33]:
%%jshell

runPy("""
# Block 12 - Create SageMaker Models (self-healing)
# This version diagnoses and fixes the common CreateModel failure modes automatically:
#   1. Verifies the role can actually read the S3 object (the real test)
#   2. Re-uploads artifacts to the SageMaker default bucket if the custom bucket is blocked
#   3. Ensures the role's inline S3 policy covers whatever bucket the artifacts live in
#   4. Waits for IAM propagation, then creates both Models
import json as _json
import time as _t
import boto3

_ROLE_NAME = execution_role.split("/")[-1]

def _role_can_read(bucket, key):
    try:
        s3.head_object(Bucket=bucket, Key=key)
        return True
    except Exception:
        return False

def _ensure_inline_s3(buckets):
    resources = []
    for b in buckets:
        resources += [f"arn:aws:s3:::{b}", f"arn:aws:s3:::{b}/*"]
    policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:PutObject", "s3:ListBucket"],
            "Resource": resources,
        }],
    }
    iam.put_role_policy(
        RoleName=_ROLE_NAME,
        PolicyName="Lesson6BucketAccess",
        PolicyDocument=_json.dumps(policy),
    )
    print(f"  Inline S3 policy set for buckets: {buckets}")

def _parse_s3(uri):
    b, k = uri.replace("s3://", "").split("/", 1)
    return b, k

def _copy_to_sm_bucket(src_uri):
    sm_bucket = f"sagemaker-{region}-{account_id}"
    # ensure bucket exists
    try:
        s3.head_bucket(Bucket=sm_bucket)
    except Exception:
        if region == "us-east-2":
            s3.create_bucket(Bucket=sm_bucket)
        else:
            s3.create_bucket(Bucket=sm_bucket,
                             CreateBucketConfiguration={"LocationConstraint": region})
    src_b, src_k = _parse_s3(src_uri)
    dst_k = f"lesson6-neo/relocated/{src_k.split('/')[-1]}"
    s3.copy_object(Bucket=sm_bucket, Key=dst_k,
                   CopySource={"Bucket": src_b, "Key": src_k})
    return f"s3://{sm_bucket}/{dst_k}", sm_bucket

if ALLOW_AWS_MUTATIONS:
    # --- Resolve artifact locations, relocating to SM bucket if needed ---
    print("Step 1: verifying role can read model artifacts...")
    sm_default_bucket = f"sagemaker-{region}-{account_id}"
    buckets_in_use = set()

    for label, uri_name in [("baseline", "baseline_s3_uri")]:
        uri = globals()[uri_name]
        b, k = _parse_s3(uri)
        if _role_can_read(b, k):
            print(f"  [{label}] readable at {uri}")
            buckets_in_use.add(b)
        else:
            print(f"  [{label}] NOT readable at {uri} - relocating to SageMaker bucket")
            new_uri, sm_b = _copy_to_sm_bucket(uri)
            globals()[uri_name] = new_uri
            buckets_in_use.add(sm_b)
            print(f"  [{label}] relocated -> {new_uri}")

    # --- Ensure the role has S3 access on all buckets actually in use ---
    print("\\nStep 2: ensuring role S3 permissions cover those buckets...")
    _ensure_inline_s3(sorted(buckets_in_use))
    print("  Waiting 90s for IAM propagation...")
    _t.sleep(90)

    # --- Create the two Models ---
    print("\\nStep 3: creating Models...")
    try:
        sm.create_model(
            ModelName=model_name_baseline,
            PrimaryContainer={
                "Image":         baseline_image,
                "ModelDataUrl":  baseline_s3_uri,
                "Environment": {
                    "SAGEMAKER_PROGRAM":             "inference.py",
                    "SAGEMAKER_REGION":              region,
                    "SAGEMAKER_CONTAINER_LOG_LEVEL": "20",
                },
            },
            ExecutionRoleArn=execution_role,
        )
        print(f"  Created Model: {model_name_baseline}")
    except ClientError as e:
        # Access the error message directly from the response for reliability
        error_message_text = e.response.get("Error", {}).get("Message", "").lower()
        if "cannot create already existing model" in error_message_text:
            print(f"  Model exists: {model_name_baseline}")
        else:
            print(f"  Baseline CreateModel failed: {e}")
            raise

    # try:
    #     sm.create_model(
    #         ModelName=model_name_neo,
    #         PrimaryContainer={
    #             "Image":         neo_image,
    #             "ModelDataUrl":  compiled_s3_uri,
    #             "Environment": {
    #                 "SAGEMAKER_REGION":              region,
    #                 "SAGEMAKER_CONTAINER_LOG_LEVEL": "20",
    #             },
    #         },
    #         ExecutionRoleArn=execution_role,
    #     )
    #     print(f"  Created Model: {model_name_neo}")
    # except ClientError as e:
    #     # Access the error message directly from the response for reliability
    #     error_message_text = e.response.get("Error", {}).get("Message", "").lower()
    #     if "cannot create already existing model" in error_message_text:
    #         print(f"  Model exists: {model_name_neo}")
    #     else:
    #         print(f"  Neo CreateModel failed: {e}")
    #         raise

    print("\\nModel created. Proceed to Block 13.")
else:
    print("Skipped (ALLOW_AWS_MUTATIONS=False).")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Step 1: verifying role can read model artifacts...
  [baseline] readable at s3://lesson6-neo-455865672536-eu-north-1/lesson6-neo/demo/models/model_baseline.tar.gz

Step 2: ensuring role S3 permissions cover those buckets...
  Inline S3 policy set for buckets: ['lesson6-neo-455865672536-eu-north-1']
  Waiting 90s for IAM propagation...

Step 3: creating Models...
  Created Model: lesson6-neo-demo-baseline-20260930-081435

Model created. Proceed to Block 13.


In [34]:
%%jshell

runPy("""
# Block 13 - Create serverless endpoint configs and endpoints (parallel)
SERVERLESS_MEMORY_MB = 2048
SERVERLESS_MAX_CONCURRENCY = 2

def _ensure_config(cfg_name, model_name):
    try:
        sm.create_endpoint_config(
            EndpointConfigName=cfg_name,
            ProductionVariants=[{
                "VariantName":          "AllTraffic",
                "ModelName":            model_name,
                "ServerlessConfig": {
                    "MemorySizeInMB": SERVERLESS_MEMORY_MB,
                    "MaxConcurrency": SERVERLESS_MAX_CONCURRENCY,
                },
            }],
        )
        print(f"  Created config: {cfg_name}")
    except ClientError as e:
        if "already exist" in str(e).lower():
            print(f"  Config exists: {cfg_name}")
        else:
            raise

def _ensure_endpoint(endpoint_name, cfg_name):
    try:
        sm.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=cfg_name)
        print(f"  Endpoint create started: {endpoint_name}")
    except ClientError as e:
        # Endpoints return "already existing" not "already exists" - match both
        if "already exist" in str(e).lower():
            sm.update_endpoint(EndpointName=endpoint_name, EndpointConfigName=cfg_name)
            print(f"  Endpoint update (blue/green) started: {endpoint_name}")
        else:
            raise

if ALLOW_AWS_MUTATIONS:
    _ensure_config(config_baseline, model_name_baseline)
    # _ensure_config(config_neo,      model_name_neo)
    _ensure_endpoint(endpoint_baseline, config_baseline)
    # _ensure_endpoint(endpoint_neo,      config_neo)
    print("\\nEndpoint call submitted. Poll in the next cell.")
else:
    print("Skipped.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ...   Created config: lesson6-neo-demo-baseline-ep-cfg-20260930-081435
  Endpoint create started: lesson6-neo-demo-baseline-ep

Endpoint call submitted. Poll in the next cell.


In [35]:
%%jshell

runPy("""
# Block 14 - Poll both endpoints until InService (~5 min each, parallel)
if ALLOW_AWS_MUTATIONS:
    endpoints_to_wait = [endpoint_baseline]
    ready = {ep: False for ep in endpoints_to_wait}
    print("Polling endpoint...")
    while not all(ready.values()):
        for ep in endpoints_to_wait:
            if ready[ep]:
                continue
            info = sm.describe_endpoint(EndpointName=ep)
            status = info["EndpointStatus"]
            marker = "READY" if status == "InService" else status
            print(f"  {dt.datetime.utcnow().isoformat(timespec='seconds')}  {ep:45s} {marker}")
            if status == "InService":
                ready[ep] = True
            elif status == "Failed":
                fr = info.get("FailureReason", "(no reason)")
                raise RuntimeError(f"{ep} failed: {fr}")
        if not all(ready.values()):
            time.sleep(30)
    print("\\nEndpoint InService.")
else:
    print("Skipped.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Polling endpoint...
  2026-09-30T08:28:43  lesson6-neo-demo-baseline-ep                  Creating
  2026-09-30T08:29:14  lesson6-neo-demo-baseline-ep                  Creating
  2026-09-30T08:29:44  lesson6-neo-demo-baseline-ep                  Creating
  2026-09-30T08:30:14  lesson6-neo-demo-baseline-ep                  Creating
  2026-09-30T08:30:44  lesson6-neo-demo-baseline-ep                  READY

Endpoint InService.


In [38]:
%%jshell

runPy("""
import boto3

REGION = 'eu-north-1'
# ENDPOINT_NAME = 'lesson6-neo-demo-neo-ep'
ENDPOINT_NAME = endpoint_baseline # Using the baseline endpoint now

sm_client = boto3.client('sagemaker', region_name=REGION)
logs_client = boto3.client('logs', region_name=REGION)

print("=====================================================")
print(" STEP 1: CHECKING DEPLOYED INSTANCE TYPE")
print("=====================================================")
try:
    ep_desc = sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
    ep_config_name = ep_desc['EndpointConfigName']
    ep_config = sm_client.describe_endpoint_config(EndpointConfigName=ep_config_name)

    if 'InstanceType' in ep_config['ProductionVariants'][0]:
        instance_type = ep_config['ProductionVariants'][0]['InstanceType']
        print(f"✅ Your endpoint is running on: {instance_type}")
        print(f"👉 CRITICAL CHECK: Go to your compilation code (where you ran .compile).")
        print(f"   If your instance_type above is 'ml.c5.xlarge', your target_instance_family MUST be 'ml_c5'.")
        print(f"   If they do not match exactly, the Neo container will silently time out.")
    elif 'ServerlessConfig' in ep_config['ProductionVariants'][0]:
        mem_size = ep_config['ProductionVariants'][0]['ServerlessConfig']['MemorySizeInMB']
        max_conc = ep_config['ProductionVariants'][0]['ServerlessConfig']['MaxConcurrency']
        print(f"✅ Your endpoint is running as serverless: MemorySizeInMB={mem_size}, MaxConcurrency={max_conc}")
        print(f"👉 Note: Serverless endpoints don't have an 'InstanceType'. Compilation target should be 'cpu'.")
    else:
        print(f"Failed to determine instance type or serverless configuration.")

except Exception as e:
    print(f"Failed to get endpoint configuration: {e}")

print("\\n=====================================================")
print(" STEP 2: PULLING HIDDEN NEO CRASH LOGS")
print("=====================================================")
log_group = f"/aws/sagemaker/Endpoints/{ENDPOINT_NAME}"
try:
    streams = logs_client.describe_log_streams(
        logGroupName=log_group,
        orderBy='LastEventTime',
        descending=True,
        limit=2
    )

    if not streams.get('logStreams'):
        print("❌ No logs found. The container failed to even start.")
    else:
        for stream in streams['logStreams']:
            print(f"\\n--- Reading recent logs from: {stream['logStreamName']} ---")
            events = logs_client.get_log_events(
                logGroupName=log_group,
                logStreamName=stream['logStreamName'],
                limit=50,
                startFromHead=False
            )
            for event in events['events']:
                msg = event['message'].strip()
                # Highlight common errors
                if "error" in msg.lower() or "exception" in msg.lower():
                    print(f"🚨 ERROR FOUND: {msg}")
                else:
                    print(msg)
except Exception as e:
    print(f"Could not fetch logs. (Check if your IAM role has CloudWatch read permissions): {e}")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... =====================================================
 STEP 1: CHECKING DEPLOYED INSTANCE TYPE
✅ Your endpoint is running as serverless: MemorySizeInMB=2048, MaxConcurrency=2
👉 Note: Serverless endpoints don't have an 'InstanceType'. Compilation target should be 'cpu'.

 STEP 2: PULLING HIDDEN NEO CRASH LOGS

--- Reading recent logs from: AllTraffic/05a288538cdbf738f6b42ccbfee42c9a-7584fd38373b45ab8408bdbe1b33c728 ---
2026-09-30T08:30:26,610 [INFO ] main org.pytorch.serve.metrics.configuration.MetricConfiguration - Successfully loaded metrics configuration from /opt/conda/lib/python3.10/site-packages/ts/configs/metrics.yaml
2026-09-30T08:30:26,721 [INFO ] main org.pytorch.serve.ModelServer -
Torchserve version: 0.11.0
TS Home: /opt/conda/lib/python3.10/site-packages
Current directory: /
Temp directory: /home/model-server/tmp
Metrics config path: /opt/conda/lib/python3.10/site-packages/ts/configs/metrics.yaml
Number of 

## 9. Smoke test + load test (above)

First a single request per endpoint to verify the invocation contract works before we hammer with 200 requests.
This catches endpoint config errors immediately instead of after minutes of load testing.

In [40]:
%%jshell

runPy("""
import time
import json
import numpy as np
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError, ReadTimeoutError
from sklearn.metrics import accuracy_score

# Ensure client uses the correct region (AWS_REGION_NAME from notebook config)
# Increased read_timeout for Neo endpoint cold start latency.
fast_config = Config(read_timeout=60, connect_timeout=5, retries={'max_attempts': 5})
fast_runtime = boto3.client("sagemaker-runtime", region_name=AWS_REGION_NAME, config=fast_config)

def _invoke_baseline(X_row):
    payload = json.dumps({"instances": X_row.tolist()})
    resp = fast_runtime.invoke_endpoint(
        EndpointName=endpoint_baseline,
        ContentType="application/json",
        Accept="application/json",
        Body=payload,
    )
    body = json.loads(resp["Body"].read().decode("utf-8"))

    # Handle various output shapes cleanly
    preds = body["predictions"] if "predictions" in body else body
    if isinstance(preds, list):
        preds = np.array(preds)

    if preds.ndim > 1 and preds.shape[1] > 1:
        return int(np.argmax(preds, axis=1)[0])
    return int(np.round(preds.ravel()[0]))

# def _invoke_neo(X_row):
#     # Pass as CSV format (most reliable payload for Neo-compiled XGBoost/SKLearn containers)
#     payload = ",".join(map(str, X_row.ravel()))

#     resp = fast_runtime.invoke_endpoint(
#         EndpointName=endpoint_neo,
#         ContentType="text/csv",
#         Accept="application/json",
#         Body=payload,
#     )

#     body_raw = resp["Body"].read().decode("utf-8")
#     try:
#         body = json.loads(body_raw)
#         preds = np.array(body["predictions"] if isinstance(body, dict) and "predictions" in body else body)
#     except Exception:
#         # Fallback for plain text float outputs
#         preds = np.fromstring(body_raw, sep=",")

#     if preds.ndim > 1 and preds.shape[1] > 1:
#         return int(np.argmax(preds, axis=1)[0])
#     return int(np.round(preds.ravel()[0]))

if ALLOW_AWS_MUTATIONS:
    print("=== Smoke test: 1 request ===")
    X_row = X_test[0:1]

    b_pred = _invoke_baseline(X_row)
    print(f"  Baseline invocation OK  -> prediction={b_pred}")

    # Increased max_retries for Neo endpoint due to potential warm-up latency
    # max_retries = 10
    # for i in range(max_retries):
    #     try:
    #         n_pred = _invoke_neo(X_row)
    #         print(f"  Neo invocation OK       -> prediction={n_pred}")
    #         break
    #     except (ReadTimeoutError, ClientError, Exception) as e:
    #         if i < max_retries - 1:
    #             print(f"  Neo warming up (attempt {i+1}/{max_retries}). Retrying in 2s...")
    #             time.sleep(2)
    #         else:
    #             raise RuntimeError(f"Neo smoke test FAILED after {max_retries} attempts: {e}") from e

    print(f"  Ground truth: y_test[0] = {y_test[0]}")
    print("\\nInvocation contract works. Safe to load-test.")
else:
    print("Skipped.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... === Smoke test: 1 request ===
  Baseline invocation OK  -> prediction=0
  Ground truth: y_test[0] = 0

Invocation contract works. Safe to load-test.


In [42]:
%%jshell

runPy("""
# Block 15 - Load-test endpoint (200 requests, batch=1)
def load_test_endpoint(invoke_fn, endpoint_label, X, y, n_runs=200):
    # Warm up (10 requests)
    for _ in range(min(10, len(X))):
        try:
            invoke_fn(X[0:1])
        except Exception:
            pass

    latencies_ms = []
    preds = []
    runs = min(n_runs, len(X))
    for i in range(runs):
        t0 = time.perf_counter()
        pred = invoke_fn(X[i:i+1])
        latencies_ms.append((time.perf_counter() - t0) * 1000)
        preds.append(pred)

    return {
        "p50_ms":         np.percentile(latencies_ms, 50),
        "p95_ms":         np.percentile(latencies_ms, 95),
        "p99_ms":         np.percentile(latencies_ms, 99),
        "throughput_rps": 1000.0 / np.mean(latencies_ms),
        "accuracy":       accuracy_score(y[:len(preds)], preds),
    }

if ALLOW_AWS_MUTATIONS:
    print("Load-testing baseline endpoint (200 requests)...")
    r_baseline = load_test_endpoint(_invoke_baseline, endpoint_baseline, X_test, y_test)
    print(f"  baseline p50: {r_baseline['p50_ms']:.2f} ms")

    # print("\\nLoad-testing Neo endpoint (200 requests)...")
    # r_neo = load_test_endpoint(_invoke_neo, endpoint_neo, X_test, y_test)
    # print(f"  neo p50:      {r_neo['p50_ms']:.2f} ms")
else:
    print("Skipped.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Load-testing baseline endpoint (200 requests)...
  baseline p50: 154.55 ms


## 10. Compare results — speedup + cost per 1k requests

In [43]:
%%jshell

runPy("""
# Block 16 - Compare + compute cost/1k requests
if ALLOW_AWS_MUTATIONS:
    df_endpoint = pd.DataFrame([r_baseline], index=[
        "Baseline (FP32 PyTorch)",
        # "Neo-compiled",
    ])
    df_endpoint = df_endpoint.round({
        "accuracy": 4, "p50_ms": 2, "p95_ms": 2, "p99_ms": 2, "throughput_rps": 1,
    })

    # ml.m5.large ~ $0.14/hr in eu-north-1 (approx; verify per your region)
    INSTANCE_COST_PER_HOUR = 0.14
    # For serverless, assume some equivalent hourly rate if it were instance-based for comparison
    # The actual serverless pricing is per inference request and duration, so this is illustrative
    # A serverless pricing model is more complex to simulate directly here.
    # For simplicity, we can use a hypothetical instance cost to show how this metric would be calculated if it were instance-based.
    # If we are purely serverless, cost_per_1k_req_usd would come directly from AWS billing for serverless invocations.
    # For now, let's just show throughput or leave this calculation commented if it's misleading.
    # For serverless, a better metric would be average cost per 1k invocations based on actual pricing: (MemorySizeInMB / 1024) * Duration_seconds * $/GB-second + $/invocation.
    # Let's keep it simple and just focus on accuracy and latency for now for a purely serverless model.

    # A more accurate calculation for serverless: (memory_in_gb * duration_seconds * cost_per_gb_second) + (number_of_invocations * cost_per_invocation)
    # We don't have the exact cost_per_gb_second and cost_per_invocation easily available here, so we will omit this for now.

    # Placeholder if we want to show a proxy for cost efficiency based on throughput
    df_endpoint["cost_proxy_per_1k_req"] = (
        (1000.0 / df_endpoint["throughput_rps"]) # Time per 1k requests in seconds
    ).round(2)


    print("=== SageMaker endpoint benchmark ===")
    print(df_endpoint.to_string())

    # p50_speedup = r_baseline["p50_ms"] / r_neo["p50_ms"]
    # p95_speedup = r_baseline["p95_ms"] / r_neo["p95_ms"]
    # cost_reduction = (1 - df_endpoint.loc["Neo-compiled", "cost_per_1k_req_usd"]
    #                     / df_endpoint.loc["Baseline (FP32 PyTorch)", "cost_per_1k_req_usd"]) * 100

    # print()
    # print(f"Neo speedup p50:  {p50_speedup:.2f}x")
    # print(f"Neo speedup p95:  {p95_speedup:.2f}x")
    # print(f"Cost reduction:   {cost_reduction:+.1f}% per 1k requests")
else:
    print("Skipped.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... === SageMaker endpoint benchmark ===
                         p50_ms  p95_ms  p99_ms  throughput_rps  accuracy  cost_proxy_per_1k_req
Baseline (FP32 PyTorch)  154.55  163.98  183.46             6.4      0.85                 156.25


## 11. Teardown — stop the billing clock

Guarded: set `CONFIRM_TEARDOWN = True` and re-run to destroy. Keeps the S3 model artifacts by default
(< 1 cent/month); flip `DELETE_S3 = True` to nuke those too.

In [45]:
%%jshell

runPy("""
# Block 17 - Dynamic Teardown (Deletes ALL matching resources)
import boto3
from botocore.exceptions import ClientError

CONFIRM_TEARDOWN = True  # Set to True to delete resources
DELETE_S3 = True  # Set to True to delete the S3 bucket
PREFIX = "lesson6-neo-demo"  # Targets all resources starting with or containing this prefix

if not CONFIRM_TEARDOWN:
    print("CONFIRM_TEARDOWN is False. Set to True and re-run to destroy.")
else:
    sm = boto3.client("sagemaker")

    # 1. Delete Endpoints
    print(">>> Deleting Endpoints")
    endpoints = sm.list_endpoints(NameContains=PREFIX).get("Endpoints", [])
    if not endpoints:
        print("  No matching endpoints found.")
    for ep in endpoints:
        ep_name = ep["EndpointName"]
        try:
            sm.delete_endpoint(EndpointName=ep_name)
            print(f"  DELETE endpoint {ep_name}")
        except ClientError as e:
            print(f"  skip endpoint {ep_name}: {e.response['Error']['Code']}")

    # 2. Delete Endpoint Configurations
    print("\\n>>> Deleting Endpoint Configs")
    configs = sm.list_endpoint_configs(NameContains=PREFIX).get(
        "EndpointConfigs", []
    )
    if not configs:
        print("  No matching configs found.")
    for cfg in configs:
        cfg_name = cfg["EndpointConfigName"]
        try:
            sm.delete_endpoint_config(EndpointConfigName=cfg_name)
            print(f"  DELETE config {cfg_name}")
        except ClientError as e:
            print(f"  skip config {cfg_name}: {e.response['Error']['Code']}")

    # 3. Delete SageMaker Models
    print("\\n>>> Deleting Models")
    models = sm.list_models(NameContains=PREFIX).get("Models", [])
    if not models:
        print("  No matching models found.")
    for m in models:
        m_name = m["ModelName"]
        try:
            sm.delete_model(ModelName=m_name)
            print(f"  DELETE model {m_name}")
        except ClientError as e:
            print(f"  skip model {m_name}: {e.response['Error']['Code']}")

    # 4. Delete S3 Bucket
    if DELETE_S3 and "bucket_name" in globals():
        print(f"\\n>>> Emptying and deleting S3 bucket: {bucket_name}")
        try:
            bucket_resource = boto3.resource("s3").Bucket(bucket_name)
            bucket_resource.object_versions.all().delete()
            bucket_resource.objects.all().delete()
            bucket_resource.delete()
            print(f"  DELETE bucket {bucket_name}")
        except ClientError as e:
            print(f"  skip bucket: {e.response['Error']['Code']}")
    else:
        print(
            f"\\nS3 bucket kept or not specified: {globals().get('bucket_name', 'N/A')}"
        )

    print("\\nTeardown complete.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... >>> Deleting Endpoints
  DELETE endpoint lesson6-neo-demo-baseline-ep

>>> Deleting Endpoint Configs
  DELETE config lesson6-neo-demo-baseline-ep-cfg-20260930-081435

>>> Deleting Models
  DELETE model lesson6-neo-demo-baseline-20260930-081435

>>> Emptying and deleting S3 bucket: lesson6-neo-455865672536-eu-north-1
  DELETE bucket lesson6-neo-455865672536-eu-north-1

Teardown complete.


## 12. Recap — What You Just Built

You executed the full Lesson 6 pipeline against real AWS:

1. **Trained locally** — worked around a training-quota=0 sandbox
2. **Exported to ONNX + INT8 quantized locally** — measured the tradeoff without AWS spend
3. **Ran a real SageMaker Neo compilation job** — the same managed service production teams use
4. **Deployed two endpoints** on identical `ml.m5.large` instances, side by side
5. **Smoke-tested and load-tested both** — caught endpoint contract errors before hammering
6. **Computed cost/1k-requests** so the tradeoff is expressed in dollars

### Next lesson

**Lesson 7 — Model Optimization II: Memory & Attention** dives into KV caching, Flash Attention, PagedAttention,
and vLLM — the LLM-specific optimizations that go beyond what Neo can do for tabular/CNN models.

In [46]:
%%jshell

runPy("""
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Current Time in IST: 2026-09-30 14:04:54
